In [ ]:
from pathlib import Path
import os
import torch
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

# Resolve project root robustly whether launched from repo root or main/.
cwd = Path.cwd()
if (cwd / 'config').exists():
    project_root = cwd
elif (cwd.parent / 'config').exists():
    project_root = cwd.parent
elif Path('/mnt/fastertalk/config').exists():
    project_root = Path('/mnt/fastertalk')
else:
    raise RuntimeError('Could not locate project root containing config/.')

os.chdir(project_root)
print('Project root:', Path.cwd())

from flame_model.FLAME import FLAMEModel
from renderer.renderer import Renderer
from pytorch3d.transforms import matrix_to_euler_angles

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
flame = FLAMEModel(n_shape=300, n_exp=50).to(device)
renderer = Renderer(render_full_head=True).to(device)
print('Renderer and FLAME ready on', device)

In [ ]:
from utils.config import load_flat_config
from models import get_model
from dataset.data_loader_joint_data_batched import get_dataloaders

cfg = load_flat_config('config/talkinghead-1kh/stage1.yaml')
cfg.batch_size = 1
cfg.save_path = '/mnt/fastertalk/logs/stage1/checkpoints/epoch_300.pt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('Checkpoint:', cfg.save_path)
print('n_embed:', cfg.n_embed)

In [ ]:
dls = get_dataloaders(cfg)
test_loader = dls['test']

def _resolve_checkpoint(path_cfg):
    path_cfg = Path(path_cfg)
    if path_cfg.is_file():
        return path_cfg
    if path_cfg.is_dir():
        ckpts = sorted(path_cfg.glob('epoch_*.pt'))
        return ckpts[-1] if ckpts else None
    return None

ckpt_file = _resolve_checkpoint(cfg.save_path)
state_dict = None
if ckpt_file is not None:
    ckpt = torch.load(ckpt_file, map_location=device)
    state_dict = ckpt['state_dict'] if isinstance(ckpt, dict) and 'state_dict' in ckpt else ckpt

model = get_model(cfg).to(device)
if state_dict is not None:
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print('Loaded checkpoint:', ckpt_file)
    print('Missing keys:', len(missing), '| Unexpected keys:', len(unexpected))
else:
    print('No valid checkpoint found at save_path - using random weights.')

In [ ]:
model.eval()
batch = next(iter(test_loader))
blendshapes_in, blendshapes_tgt, mask = batch
blendshapes_in = blendshapes_in.to(device)
blendshapes_tgt = blendshapes_tgt.to(device)
mask = mask.to(device)

with torch.no_grad():
    pred, qloss = model(blendshapes_in, mask)

valid = mask.unsqueeze(-1).float()
mse = (((pred[..., :-2] - blendshapes_tgt[..., :-2]) ** 2) * valid).sum() / torch.clamp(valid.sum() * pred[..., :-2].shape[-1], min=1.0)

print('Input shape:', tuple(blendshapes_in.shape))
print('Pred shape:', tuple(pred.shape))
print('Quant loss:', float(qloss.mean().item()))
print('Masked MSE:', float(mse.item()))

In [ ]:
def get_vertices_from_blendshapes(expr, gpose, jaw, eyelids):
    expr_tensor = expr.to(device)
    gpose_tensor = gpose.to(device)
    jaw_tensor = jaw.to(device)
    _ = eyelids.to(device)  # kept for parity with dataset layout
    
    target_shape_tensor = torch.zeros(expr_tensor.shape[0], 300, device=device)
    I = matrix_to_euler_angles(torch.cat([torch.eye(3, device=device)[None]], dim=0), 'XYZ')
    eye_r = I.clone().squeeze()
    eye_l = I.clone().squeeze()
    eyes = torch.cat([eye_r, eye_l], dim=0).expand(expr_tensor.shape[0], -1)
    pose = torch.cat([gpose_tensor, jaw_tensor], dim=-1)
    
    verts, _ = flame.forward(
        shape_params=target_shape_tensor,
        expression_params=expr_tensor,
        pose_params=pose,
        eye_pose_params=eyes,
    )
    return verts.detach()


def render_comparison(blendshapes_gt, blendshapes_pred, index, out_dir='demo/video'):
    expr_gt = blendshapes_gt[:, :50]
    gpose_gt = blendshapes_gt[:, 50:53]
    jaw_gt = blendshapes_gt[:, 53:56]
    eyelids_gt = blendshapes_gt[:, 56:]

    expr_pr = blendshapes_pred[:, :50]
    gpose_pr = blendshapes_pred[:, 50:53]
    jaw_pr = blendshapes_pred[:, 53:56]
    eyelids_pr = blendshapes_pred[:, 56:]

    verts_gt = get_vertices_from_blendshapes(expr_gt, gpose_gt, jaw_gt, eyelids_gt)
    verts_pr = get_vertices_from_blendshapes(expr_pr, gpose_pr, jaw_pr, eyelids_pr)

    cam = torch.tensor([5, 0, 0], dtype=torch.float32, device=device).unsqueeze(0)
    cam = cam.expand(verts_gt.shape[0], -1)

    frames_gt = renderer.forward(verts_gt, cam)['rendered_img']
    frames_pr = renderer.forward(verts_pr, cam)['rendered_img']

    os.makedirs(out_dir, exist_ok=True)
    video_file = os.path.join(out_dir, f'sample_{index:03d}.mp4')

    def update(frame_idx, gt_seq, pr_seq, ax):
        gt = gt_seq[frame_idx].detach().cpu().numpy().transpose(1, 2, 0)
        pr = pr_seq[frame_idx].detach().cpu().numpy().transpose(1, 2, 0)
        combined = np.concatenate([(gt * 255).astype(np.uint8), (pr * 255).astype(np.uint8)], axis=1)
        ax.clear()
        ax.imshow(combined)
        ax.axis('off')

    fig, ax = plt.subplots(figsize=(10, 5))
    ani = animation.FuncAnimation(
        fig,
        update,
        frames=frames_gt.shape[0],
        fargs=(frames_gt, frames_pr, ax),
        interval=100,
    )
    ani.save(video_file, writer='ffmpeg', fps=25)
    plt.close(fig)
    print('Saved video comparison to', video_file)


def evaluate_samples(model, data_loader, num_samples=3):
    model.eval()
    for i, batch in enumerate(data_loader):
        if i >= num_samples:
            break

        blendshapes_in, blendshapes_tgt, mask = batch
        blendshapes_in = blendshapes_in.to(device)
        blendshapes_tgt = blendshapes_tgt.to(device)
        mask = mask.to(device)

        with torch.no_grad():
            pred, qloss = model(blendshapes_in, mask)

        valid = mask.unsqueeze(-1).float()
        mse = (((pred[..., :-2] - blendshapes_tgt[..., :-2]) ** 2) * valid).sum() / torch.clamp(
            valid.sum() * pred[..., :-2].shape[-1], min=1.0
        )

        print(f'sample={i} quant={float(qloss.mean().item()):.6f} mse={float(mse.item()):.6f}')
        render_comparison(blendshapes_tgt.squeeze(0), pred.squeeze(0), i)

In [ ]:
# Render a few test samples as input-vs-output videos.
evaluate_samples(model, test_loader, num_samples=3)